# Notebook 05: Momentum Contrast (MoCo)

## Why This Matters
SimCLR's quality scales with batch size because more images in a batch means more negatives. Getting to 4096 requires massive GPU memory or TPU clusters. MoCo (He et al., 2020) decouples negative count from batch size using two simple ideas: a **momentum encoder** and a **key queue**. The result: you get 65,536 negatives per positive at the compute cost of a standard batch.

MoCo also introduced the momentum encoder pattern that DINO later adapted for self-distillation — understanding it here makes DINO immediately legible in Notebook 06.

## Structure
1. The SimCLR batch-size problem (narrative)
2. The momentum encoder — why slow updates work
3. The key queue — decoupling negative count from batch size
4. MoCo training loop
5. MoCo v2 improvements (narrative)
6. Bridge to DINO

## What You'll Learn
- How exponential moving average updates create a stable key encoder
- Why queue consistency requires momentum (vs random resets)
- How MoCo's dictionary-as-a-queue gives 65k negatives at standard batch sizes
- The MoCo v2 improvements: MLP projection head + stronger augmentation


## Background

### The SimCLR problem

SimCLR treats all other images in a batch as negatives. With N=256:
- 510 negatives per positive — decent
- Requires 256 × 2 full forward passes through the encoder
- Both views need gradients → 2× memory cost

With N=4096 (SimCLR's optimal):
- 8190 negatives — excellent quality
- Requires 512 TPUs or equivalent — not accessible

### MoCo's solution: momentum encoder + queue

**Key queue:** Maintain a FIFO queue of K encoded representations (K=65536). For each training step, enqueue the current batch's encodings and dequeue the oldest. Every query now has 65,536 negatives regardless of batch size.

**Momentum encoder:** The queue is useless if the encodings drift — a key encoded 1000 steps ago under different weights is inconsistent with current queries. MoCo solves this with a slowly-updating momentum encoder:

```
θ_k ← m × θ_k + (1-m) × θ_q      # m ≈ 0.999
```

The query encoder (θ_q) updates via gradients normally. The key encoder (θ_k) updates via exponential moving average — slow enough that queue entries remain consistent. Critically: **no gradients flow through the key encoder**, which also saves memory.

### MoCo v1 vs v2

MoCo v1 (2020): linear projection, standard augmentations, InfoNCE loss.
MoCo v2 (2020): added SimCLR's MLP projection head + stronger augmentations. Matched SimCLR quality with far less compute. This version is worth implementing.


In [ ]:
# %pip install -q torch torchvision numpy matplotlib scikit-learn

In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

torch.manual_seed(42)
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 1. Augmentation & Dataset

In [ ]:
class MoCoAugmentation:
    """MoCo v2 augmentation — stronger than v1, matches SimCLR's."""
    def __init__(self, img_size=32):
        self.transform_q = T.Compose([  # query augmentation
            T.RandomResizedCrop(img_size, scale=(0.2, 1.0)),
            T.RandomHorizontalFlip(),
            T.RandomApply([T.ColorJitter(0.4,0.4,0.4,0.1)], p=0.8),
            T.RandomGrayscale(p=0.2),
            T.ToTensor(),
            T.Normalize([0.4914,0.4822,0.4465],[0.2023,0.1994,0.2010]),
        ])
        self.transform_k = T.Compose([  # key augmentation (same pipeline)
            T.RandomResizedCrop(img_size, scale=(0.2, 1.0)),
            T.RandomHorizontalFlip(),
            T.RandomApply([T.ColorJitter(0.4,0.4,0.4,0.1)], p=0.8),
            T.RandomGrayscale(p=0.2),
            T.ToTensor(),
            T.Normalize([0.4914,0.4822,0.4465],[0.2023,0.1994,0.2010]),
        ])

    def __call__(self, x):
        return self.transform_q(x), self.transform_k(x)


class MoCoDataset(Dataset):
    def __init__(self, dataset, aug):
        self.dataset = dataset
        self.aug = aug
    def __len__(self): return len(self.dataset)
    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        xq, xk = self.aug(img)
        return xq, xk, label


cifar_base = torchvision.datasets.CIFAR10("/tmp/cifar10", train=True, download=True,
                                           transform=torchvision.transforms.ToTensor())
moco_dataset = MoCoDataset(cifar_base, MoCoAugmentation())
moco_loader  = DataLoader(moco_dataset, batch_size=128, shuffle=True,
                          num_workers=0, drop_last=True)
print(f"Dataset: {len(moco_dataset)} images, batch_size=128")

## 2. MoCo Model — Encoder + Momentum Encoder + Queue

In [ ]:
class MoCo(nn.Module):
    """MoCo v2: momentum encoder + key queue + MLP projection head."""
    def __init__(self, base_dim=512, proj_dim=128, queue_size=4096, momentum=0.999, temperature=0.2):
        super().__init__()
        self.K   = queue_size
        self.m   = momentum
        self.tau = temperature

        # Query encoder (updated by gradients)
        self.encoder_q = self._build_encoder(base_dim, proj_dim)
        # Key encoder (updated by momentum — no gradients)
        self.encoder_k = self._build_encoder(base_dim, proj_dim)

        # Initialize key encoder weights == query encoder weights
        for pq, pk in zip(self.encoder_q.parameters(), self.encoder_k.parameters()):
            pk.data.copy_(pq.data)
            pk.requires_grad = False  # key encoder never receives gradients

        # Queue: [proj_dim, K] — stored as columns for efficient bmm
        self.register_buffer("queue", F.normalize(torch.randn(proj_dim, queue_size), dim=0))
        self.register_buffer("queue_ptr", torch.zeros(1, dtype=torch.long))

    def _build_encoder(self, base_dim, proj_dim):
        resnet = torchvision.models.resnet18(weights=None)
        resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        resnet.maxpool = nn.Identity()
        backbone = nn.Sequential(*list(resnet.children())[:-1])
        projector = nn.Sequential(
            nn.Linear(base_dim, base_dim), nn.ReLU(inplace=True),
            nn.Linear(base_dim, proj_dim)
        )
        return nn.Sequential(backbone, nn.Flatten(), projector)

    @torch.no_grad()
    def _momentum_update(self):
        """Slowly update key encoder weights toward query encoder."""
        for pq, pk in zip(self.encoder_q.parameters(), self.encoder_k.parameters()):
            pk.data = pk.data * self.m + pq.data * (1.0 - self.m)

    @torch.no_grad()
    def _enqueue_dequeue(self, keys: torch.Tensor):
        """Push new keys into queue, drop oldest."""
        batch_size = keys.shape[0]
        ptr = int(self.queue_ptr)
        # Overwrite queue positions (wraps around)
        self.queue[:, ptr:ptr + batch_size] = keys.T
        self.queue_ptr[0] = (ptr + batch_size) % self.K

    def forward(self, xq: torch.Tensor, xk: torch.Tensor):
        # Encode queries (with gradients)
        q = F.normalize(self.encoder_q(xq), dim=1)  # [N, D]

        # Encode keys (no gradients through key encoder)
        with torch.no_grad():
            self._momentum_update()
            k = F.normalize(self.encoder_k(xk), dim=1)  # [N, D]

        # Positive logits: each query's dot product with its own key
        l_pos = torch.einsum("nd,nd->n", q, k).unsqueeze(1)  # [N, 1]

        # Negative logits: each query vs all queued keys
        l_neg = torch.mm(q, self.queue.clone().detach())       # [N, K]

        logits = torch.cat([l_pos, l_neg], dim=1) / self.tau  # [N, 1+K]

        # Positive is always index 0
        targets = torch.zeros(logits.shape[0], dtype=torch.long, device=logits.device)
        loss = F.cross_entropy(logits, targets)

        # Update queue with current keys
        self._enqueue_dequeue(k)

        return loss

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Encoder output before projection (for downstream tasks)."""
        resnet_part = self.encoder_q[0]  # backbone
        with torch.no_grad():
            return resnet_part(x).squeeze(-1).squeeze(-1)


moco = MoCo(base_dim=512, proj_dim=128, queue_size=4096, momentum=0.999, temperature=0.2).to(device)
print(f"MoCo parameters: {sum(p.numel() for p in moco.parameters() if p.requires_grad):,} trainable")
print(f"Queue size: {moco.K} negatives per positive")
print(f"Momentum: {moco.m} (key encoder update rate)")
print(f"Temperature: {moco.tau}")

# Forward pass check
xq_t = torch.randn(4, 3, 32, 32, device=device)
xk_t = torch.randn(4, 3, 32, 32, device=device)
loss_t = moco(xq_t, xk_t)
print(f"\nForward pass loss: {loss_t.item():.4f}")

## 3. Training Loop

In [ ]:
def train_moco(model, loader, epochs=5, lr=0.03):
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    losses = []
    model.train()

    for epoch in range(epochs):
        epoch_loss, n = 0.0, 0
        for xq, xk, _ in loader:
            xq, xk = xq.to(device), xk.to(device)
            loss = model(xq, xk)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item(); n += 1
        scheduler.step()
        avg = epoch_loss / n
        losses.append(avg)
        print(f"Epoch {epoch+1}/{epochs}  loss: {avg:.4f}")

    return losses


print("Training MoCo for 5 epochs (demo — production uses 200 epochs)")
losses = train_moco(moco, moco_loader, epochs=5, lr=0.03)

plt.figure(figsize=(7,3))
plt.plot(losses, marker='o')
plt.xlabel("Epoch"); plt.ylabel("InfoNCE Loss")
plt.title("MoCo Training Loss")
plt.tight_layout(); plt.show()

## 4. SimCLR vs MoCo — Side by Side

In [ ]:
comparison = {
    "Negatives per positive": ("2(N-1) = 254 at N=128", "Queue K = 4096 at any batch size"),
    "Negative source":        ("In-batch (current step only)", "Queue (accumulated across steps)"),
    "Key encoder":            ("Same as query, full gradients", "Momentum copy, no gradients"),
    "Memory cost":            ("2× encoder in memory", "1× query + 1× frozen key"),
    "Batch size required":    ("Large (4096 for best quality)", "Standard (128-256 works well)"),
    "Compute":                ("2 full forward passes w/ grad", "1 grad pass + 1 no-grad pass"),
    "Queue staleness":        ("None (fresh each step)", "Slight (old keys under old weights)"),
    "Staleness mitigation":   ("N/A", "Momentum update keeps encoders close"),
}

print(f"{'Aspect':<30} {'SimCLR':<35} {'MoCo'}")
print("-" * 90)
for aspect, (simclr, moco_v) in comparison.items():
    print(f"{aspect:<30} {simclr:<35} {moco_v}")

print()
print("Bottom line: MoCo achieves similar representation quality to SimCLR")
print("with ~4× fewer GPUs and standard batch sizes.")

## 5. MoCo v2 and Beyond (Narrative)

**MoCo v2** (Chen et al., 2020b) added SimCLR's MLP projection head and stronger augmentations, matching SimCLR quality. This is the version worth using in practice.

**MoCo v3** (Chen et al., 2021) replaced the ResNet backbone with a Vision Transformer (ViT), removed the queue (large ViT batches provide enough negatives), and added a stop-gradient trick borrowed from BYOL. By MoCo v3, the momentum encoder is doing something different than originally intended — it's closer to BYOL's EMA target network than MoCo v1's consistency trick.

The progression: MoCo v1 → solved the batch size problem. MoCo v2 → matched SimCLR quality. MoCo v3 → transitioned to ViT-native self-supervised learning. DINO is where this line of thinking culminates for vision.


## 6. Bridge to DINO

MoCo introduced the momentum encoder pattern: a slowly-updated copy of the query encoder that produces stable targets. DINO repurposes this exact mechanism for **self-distillation** — but without a contrastive loss or explicit negatives.

DINO's teacher network is the momentum encoder. DINO's student network is the query encoder. Instead of contrasting against a queue of negatives, DINO trains the student to match the teacher's output distribution via cross-entropy. The centering operation (subtracting a running mean from teacher outputs) prevents collapse — the role that negative examples play in MoCo.

The result: emergent segmentation properties, attention maps that find object boundaries without supervision, and strong linear evaluation scores — from the same momentum encoder idea you just implemented.

Notebook 06 covers DINO's feature extraction and zero-shot segmentation properties.


## Summary

| Concept | Key Point |
|---------|-----------|
| Momentum encoder | EMA of query encoder: `θ_k ← m·θ_k + (1-m)·θ_q`, m≈0.999 |
| Key queue | FIFO buffer of K encoded keys — negatives from past batches |
| No queue gradients | Key encoder frozen to gradient flow → memory efficient |
| Queue consistency | Momentum update keeps old queue entries consistent with current encoder |
| vs SimCLR | Same quality, 4× less GPU; works at batch_size=128 vs 4096 |
| MoCo v2 | Added MLP projection head + stronger augmentation → matches SimCLR |

**Next:** Notebook 06 — DINO. Self-distillation with no explicit negatives: the teacher is a momentum encoder, the student matches its distribution, centering prevents collapse. Emergent segmentation falls out for free.
